# Mechanism-isolation ablation simulation

Runs the simulator and regenerates `demos/scheduler_tests/simulation_results.json` and `simulation results.md` from template.

A: baseline fixed-cap FIFO unsorted  
B: sort-only fixed-cap (same n)  
C: sort + DP + FIFO  
D: sort + DP + MINMAX  
E: sort + DP + BATCH_SIZE  
F: sort + greedy batch-construction + MINMAX


In [1]:
from pathlib import Path
import json
import subprocess
import pandas as pd

def _repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'PAPER.md').exists() and (p / 'demos' / 'scheduler_tests').exists():
            return p
    raise RuntimeError('Could not find repository root from current working directory.')

repo_root = _repo_root(Path.cwd().resolve())
subprocess.run(['python', 'demos/scheduler_tests/generate_simulation_results.py'], cwd=repo_root, check=True)
results = json.loads((repo_root / 'demos/scheduler_tests/simulation_results.json').read_text())
print('Regenerated simulation artifacts from template-backed generator.')


Regenerated simulation artifacts from template-backed generator.


In [2]:
summary_rows=[]
for w,r in results['workloads'].items():
    m=r['mechanism_isolation_ablation']
    g=m['incremental_rps_gain']
    summary_rows.append({
        'workload': w,
        'fixed_n': m['fixed_n_for_isolation'],
        'sort_only_minus_baseline': g['sort_only_minus_baseline'],
        'dp_minmax_minus_sort_only': g['dp_minmax_minus_sort_only'],
        'dp_fifo_minus_dp_minmax': g['dp_fifo_minus_dp_minmax'],
        'batch_size_minus_dp_minmax': g['batch_size_minus_dp_minmax'],
        'greedy_minmax_minus_dp_minmax': g['greedy_minmax_minus_dp_minmax'],
    })
pd.DataFrame(summary_rows)


,workload,fixed_n,sort_only_minus_baseline,dp_minmax_minus_sort_only,dp_fifo_minus_dp_minmax,batch_size_minus_dp_minmax,greedy_minmax_minus_dp_minmax
0,synthetic,1,0.000000,1.712412,-0.005743,0.034482,-0.107629
1,gpu_bge_m3,3,0.037458,9.248643,0.317228,-0.572158,-5.804547
2,cpu_jina,1,0.000000,0.882479,-0.000310,0.002589,-0.249562


In [3]:
for w,r in results['workloads'].items():
    print(f'\n=== {w} mechanism variants ===')
    variants=r['mechanism_isolation_ablation']['variants']
    for name,v in variants.items():
        rps=[round(x['throughput_rps'],3) for x in v['raw_trials']]
        p95=[round(x['p95_latency_s'],3) for x in v['raw_trials']]
        print(name, 'RPS=', rps, 'P95=', p95)



=== synthetic mechanism variants ===
A_baseline_fifo_unsorted RPS= [11.427, 11.231, 11.38, 11.318, 11.382] P95= [1.789, 1.823, 1.792, 1.812, 1.802]
B_sort_only_fixed_cap RPS= [11.427, 11.231, 11.38, 11.318, 11.382] P95= [0.202, 0.202, 0.202, 0.201, 0.201]
C_sort_plus_dp_fifo RPS= [13.163, 12.893, 13.095, 13.017, 13.105] P95= [2.255, 2.283, 2.266, 2.289, 2.262]
D_sort_plus_dp_minmax RPS= [13.173, 12.903, 13.1, 13.018, 13.106] P95= [2.302, 2.349, 2.301, 2.306, 2.323]
E_sort_plus_dp_batch_size RPS= [13.203, 12.934, 13.141, 13.055, 13.14] P95= [3.165, 3.215, 3.184, 3.194, 3.186]
F_sort_plus_greedy_minmax RPS= [13.058, 12.794, 12.996, 12.913, 13.001] P95= [2.354, 2.41, 2.34, 2.361, 2.366]

=== gpu_bge_m3 mechanism variants ===
A_baseline_fifo_unsorted RPS= [27.906, 27.813, 27.886, 27.887, 27.941] P95= [0.691, 0.693, 0.693, 0.692, 0.69]
B_sort_only_fixed_cap RPS= [27.979, 27.85, 27.916, 27.863, 28.011] P95= [0.138, 0.138, 0.137, 0.137, 0.137]
C_sort_plus_dp_fifo RPS= [37.628, 37.253, 37.581